In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install implicit

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

transactions = pd.read_parquet('/kaggle/input/notebooks/hahahaha34234/phase-1/transactions_clean.parquet')
articles = pd.read_parquet('/kaggle/input/notebooks/hahahaha34234/phase-3/articles_with_style.parquet')
customer_segments = pd.read_parquet('/kaggle/input/notebooks/hahahaha34234/phase-5/customer_segments_scored.parquet')

transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print(f"Transactions: {transactions.shape}")
print(f"Unique customers: {transactions['customer_id'].nunique()}")
print(f"Unique articles: {transactions['article_id'].nunique()}")

In [ ]:
# Use last 4 weeks as test set, everything before as train
cutoff_date = pd.to_datetime('2020-08-25')

train_data = transactions[transactions['t_dat'] < cutoff_date].copy()
test_data = transactions[transactions['t_dat'] >= cutoff_date].copy()

print(f"Train transactions: {len(train_data)}")
print(f"Test transactions: {len(test_data)}")
print(f"Test date range: {test_data['t_dat'].min()} to {test_data['t_dat'].max()}")

# Only evaluate on customers who appear in both train and test
train_customers = set(train_data['customer_id'].unique())
test_customers = set(test_data['customer_id'].unique())
eval_customers = train_customers.intersection(test_customers)

print(f"Customers in both train and test: {len(eval_customers)}")

In [ ]:
# Map customer and article IDs to consecutive integers
all_customers = train_data['customer_id'].unique()
all_articles = train_data['article_id'].unique()

customer2idx = {c: i for i, c in enumerate(all_customers)}
article2idx = {a: i for i, a in enumerate(all_articles)}
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

train_data['customer_idx'] = train_data['customer_id'].map(customer2idx)
train_data['article_idx'] = train_data['article_id'].map(article2idx)

# Drop rows where mapping failed
train_data.dropna(subset=['customer_idx', 'article_idx'], inplace=True)
train_data['customer_idx'] = train_data['customer_idx'].astype(int)
train_data['article_idx'] = train_data['article_idx'].astype(int)

print(f"Customers in train: {len(all_customers)}")
print(f"Articles in train: {len(all_articles)}")

In [ ]:
# Count how many times each customer bought each article
# This is your confidence signal - buying something 3 times is stronger than buying once
interaction_counts = (train_data.groupby(['customer_idx', 'article_idx'])
                      .size()
                      .reset_index(name='purchase_count'))

# Build sparse matrix - rows are articles, columns are customers
# implicit library expects item x user format
n_articles = len(all_articles)
n_customers = len(all_customers)

item_user_matrix = csr_matrix((
    interaction_counts['purchase_count'].values,
    (interaction_counts['article_idx'].values, interaction_counts['customer_idx'].values)
), shape=(n_articles, n_customers))

print(f"Item-user matrix shape: {item_user_matrix.shape}")
print(f"Matrix density: {item_user_matrix.nnz / (n_articles * n_customers):.6f}")
print(f"Non-zero entries: {item_user_matrix.nnz:,}")

assert item_user_matrix.shape[0] == n_articles
assert item_user_matrix.shape[1] == n_customers

In [ ]:
# # ALS - Alternating Least Squares
# # factors = dimensionality of the latent space
# # iterations = number of alternating optimization steps
# # regularization = prevents overfitting on sparse data

# model = implicit.als.AlternatingLeastSquares(
#     factors=64,
#     iterations=20,
#     regularization=0.05,
#     alpha=40,        # Confidence scaling - how much to trust purchase counts
#     random_state=42
# )

# # implicit expects item x user matrix
# model.fit(item_user_matrix)

# print("ALS model trained successfully")
# print(f"User factors shape: {model.user_factors.shape}")
# print(f"Item factors shape: {model.item_factors.shape}")

In [ ]:
# ── RETRAIN with explicit user=customers, item=articles convention ──────────

# user x item matrix (customers as rows, articles as columns)
# We will pass this directly to implicit which internally treats rows as users
user_item_matrix = csr_matrix((
    interaction_counts['purchase_count'].values,
    (interaction_counts['customer_idx'].values, interaction_counts['article_idx'].values)
), shape=(n_customers, n_articles))

print(f"User-item matrix shape (customers x articles): {user_item_matrix.shape}")
assert user_item_matrix.shape[0] == n_customers
assert user_item_matrix.shape[1] == n_articles

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.05,
    alpha=40,
    random_state=42
)

# implicit.fit() expects item x user — so pass the transpose here
model.fit(user_item_matrix)

print(f"User factors shape (should be n_customers x 64): {model.user_factors.shape}")
print(f"Item factors shape (should be n_articles x 64):  {model.item_factors.shape}")

# Verify shapes match expectations
assert model.user_factors.shape[0] == n_customers, "User factors rows should equal n_customers"
assert model.item_factors.shape[0] == n_articles,  "Item factors rows should equal n_articles"
print("Shape assertions passed — model is oriented correctly")

In [ ]:
# Precompute user-item matrix (transpose) once - not inside the function
user_item_matrix = item_user_matrix.T.tocsr()  # shape: customers x articles

#updated
def get_recommendations_for_customer(customer_id, n=12):
    if customer_id not in customer2idx:
        return []
    
    customer_idx = customer2idx[customer_id]
    
    # user_item_matrix row for this customer (1 x n_articles)
    user_row = user_item_matrix[customer_idx]
    
    recommended_ids, scores = model.recommend(
        customer_idx,
        user_row,
        N=n,
        filter_already_liked_items=True
    )
    
    # recommended_ids are article indices — decode to article_id strings
    return [idx2article[int(idx)] for idx in recommended_ids if int(idx) in idx2article]

# Sanity check
test_data['article_id'] = test_data['article_id'].astype(str)
idx2article = {i: str(a) for a, i in article2idx.items()}

test_actual_sample = (test_data[test_data['customer_id'].isin(list(eval_customers)[:5])]
                      .groupby('customer_id')['article_id']
                      .apply(list).to_dict())

for cid, actual in test_actual_sample.items():
    recs = get_recommendations_for_customer(cid, n=12)
    hits = len(set(recs) & set(actual))
    print(f"Customer {cid[:10]}... | Actual: {len(actual)} | Recs: {len(recs)} | Hits: {hits}")

# Test it on one customer
sample_customer = list(eval_customers)[0]
recs = get_recommendations_for_customer(sample_customer, n=12)
print(f"Sample recommendations for customer {sample_customer[:10]}...")
print(f"Recommended article IDs: {recs[:5]}")

# Show what these articles actually are
rec_details = articles[articles['article_id'].isin(recs)][
    ['article_id', 'product_type_name', 'product_group_name', 'style_group']
]
print(rec_details)

In [ ]:
def average_precision_at_k(recommended, actual, k=12):
    if not actual:
        return 0.0
    
    recommended = recommended[:k]
    hits = 0
    precision_sum = 0.0
    
    for i, item in enumerate(recommended):
        if item in actual:
            hits += 1
            precision_sum += hits / (i + 1)
    
    return precision_sum / min(len(actual), k)

def evaluate_map_at_k(eval_customer_ids, k=12):
    ap_scores = []
    
    # Build ground truth from test set
    test_actual = (test_data[test_data['customer_id'].isin(eval_customer_ids)]
                   .groupby('customer_id')['article_id']
                   .apply(list)
                   .to_dict())
    
    for customer_id in eval_customer_ids:
        if customer_id not in test_actual:
            continue
        actual_items = test_actual[customer_id]
        recommended_items = get_recommendations_for_customer(customer_id, n=k)
        ap = average_precision_at_k(recommended_items, actual_items, k=k)
        ap_scores.append(ap)
    
    return np.mean(ap_scores) if ap_scores else 0.0

# --- FIX TYPE MISMATCH ---
test_data['article_id'] = test_data['article_id'].astype(str)
idx2article = {i: str(a) for a, i in article2idx.items()}

# Sanity check on 5 customers before running full MAP evaluation
test_actual_sample = (test_data[test_data['customer_id'].isin(list(eval_customers)[:5])]
                      .groupby('customer_id')['article_id']
                      .apply(list).to_dict())

for cid, actual in test_actual_sample.items():
    recs = get_recommendations_for_customer(cid, n=12)
    hits = len(set(recs) & set(actual))
    print(f"Customer {cid[:10]}... | Actual: {len(actual)} | Recs: {len(recs)} | Hits: {hits}")

# Evaluate on a sample of eval customers to keep runtime manageable
eval_sample = list(eval_customers)[:2000]
map12 = evaluate_map_at_k(eval_sample, k=12)
print(f"MAP@12 on {len(eval_sample)} customers: {map12:.4f}")

In [ ]:
len(set(train_data['article_id']) & set(test_data['article_id']))

In [ ]:
def get_cold_start_recommendations(customer_id, n=12):
    # Look up customer style affinity
    customer_row = customer_segments[customer_segments['customer_id'] == customer_id]
    
    if customer_row.empty or pd.isnull(customer_row['style_affinity'].values[0]):
        # No style info either - return globally popular items
        popular = (train_data.groupby('article_id')['customer_id']
                   .count()
                   .sort_values(ascending=False)
                   .head(n).index.tolist())
        return popular
    
    style = customer_row['style_affinity'].values[0]
    
    # Recommend top selling articles from their style group
    style_articles = articles[articles['style_group'] == style]['article_id'].tolist()
    style_transactions = train_data[train_data['article_id'].isin(style_articles)]
    
    top_style_items = (style_transactions.groupby('article_id')['customer_id']
                       .count()
                       .sort_values(ascending=False)
                       .head(n).index.tolist())
    
    return top_style_items

def hybrid_recommend(customer_id, n=12):
    # Try ALS first, fall back to cold start
    if customer_id in customer2idx:
        return get_recommendations_for_customer(customer_id, n=n)
    else:
        return get_cold_start_recommendations(customer_id, n=n)

# Test hybrid recommender
print("Testing hybrid recommender:")
print(f"ALS recommendation: {get_recommendations_for_customer(sample_customer, n=3)}")

fake_new_customer = 'new_customer_xyz'
print(f"Cold start recommendation: {get_cold_start_recommendations(fake_new_customer, n=3)}")

In [ ]:
print("=" * 60)
print("PHASE 6 SUMMARY - RECOMMENDER SYSTEM")
print("=" * 60)
print(f"Model: Alternating Least Squares (ALS)")
print(f"Training customers: {len(all_customers):,}")
print(f"Training articles: {len(all_articles):,}")
print(f"Matrix density: {item_user_matrix.nnz / (n_articles * n_customers):.6f}")
print(f"Latent factors: 64")
print(f"MAP@12 (2000 customer sample): {map12:.4f}")
print(f"Cold start strategy: Style affinity fallback from NLP clustering")
print(f"System type: Hybrid (collaborative + content-based)")

In [ ]:
# Save recommendations for the at risk cohort specifically
at_risk_customers = customer_segments[
    customer_segments['segment'] == 'At Risk'
]['customer_id'].tolist()

at_risk_recs = []
for cid in at_risk_customers[:5000]:
    recs = hybrid_recommend(cid, n=12)
    at_risk_recs.append({'customer_id': cid, 'recommendations': recs})

at_risk_recs_df = pd.DataFrame(at_risk_recs)
at_risk_recs_df.to_parquet('/kaggle/working/at_risk_recommendations.parquet', index=False)

print(f"Saved at_risk_recommendations.parquet")
print(f"At Risk customers with recommendations: {len(at_risk_recs_df)}")
print(at_risk_recs_df.head(3))